In [3]:
print("""
@File         : Marking imputed values.ipynb
@Author(s)    : Stephen CUI
@LastEditor(s): Stephen CUI
@CreatedTime  : 2025-01-26 15:45:48
@Email        : cuixuanstephen@gmail.com
@Description  : 标记插补值
""")


@File         : Marking imputed values.ipynb
@Author(s)    : Stephen CUI
@LastEditor(s): Stephen CUI
@CreatedTime  : 2025-01-26 15:45:48
@Email        : cuixuanstephen@gmail.com
@Description  : 标记插补值



In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from feature_engine.imputation import AddMissingIndicator, CategoricalImputer, MeanMedianImputer

In [5]:
data = pd.read_csv("../../DATA/credit_approval_uci.csv")
X_train, X_test, y_train, y_test = train_test_split(
    data.drop("target", axis=1),
    data["target"],
    test_size=0.3,
    random_state=0,
)

In [6]:
varnames = ["A1", "A3", "A4", "A5", "A6", "A7", "A8"]
indicators = [f'{var}_na' for var in varnames]

In [7]:
X_train_t = X_train.copy()
X_test_t = X_test.copy()

In [8]:
X_train_t[indicators] = X_train[varnames].isna().astype(int)
X_test_t[indicators] = X_test[varnames].isna().astype(int)

In [9]:
X_train_t.sample(2)

,A1,A2,A3,A4,A5,A6,A7,A8,A9,A10,A11,A12,A13,A14,A15,A1_na,A3_na,A4_na,A5_na,A6_na,A7_na,A8_na
86,b,NaN,0.375,u,g,d,v,0.875,t,f,0,t,s,928.0,0,0,0,0,0,0,0,0
438,a,27.17,NaN,u,g,ff,ff,NaN,NaN,NaN,1,f,g,92.0,300,0,1,0,0,0,0,1


In [10]:
imputer = AddMissingIndicator(variables=None, missing_only=True)

In [11]:
imputer.fit(X_train)

AddMissingIndicator()

In [12]:
imputer.variables_

['A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8', 'A9', 'A10', 'A14']

In [13]:
X_train_t = imputer.transform(X_train)
X_test_t = imputer.transform(X_test)

In [14]:
X_train_t.sample(2)

,A1,A2,A3,A4,A5,A6,A7,A8,A9,A10,A11,A12,A13,A14,A15,A1_na,A2_na,A3_na,A4_na,A5_na,A6_na,A7_na,A8_na,A9_na,A10_na,A14_na
576,b,30.17,NaN,u,g,c,v,NaN,NaN,NaN,11,f,g,32.0,540,0,0,1,0,0,0,0,1,1,1,0
622,a,25.58,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,f,p,NaN,0,0,0,1,1,1,1,1,1,1,1,1


In [15]:
pipe = Pipeline(
    [('indicators', AddMissingIndicator(missing_only=True)),
     ('categorical', CategoricalImputer(imputation_method='frequent')),
     ('numerical', MeanMedianImputer())]
)

In [16]:
X_train_t = pipe.fit_transform(X_train)
X_test_t = pipe.transform(X_test)

In [17]:
X_train_t.sample(2)

,A1,A2,A3,A4,A5,A6,A7,A8,A9,A10,A11,A12,A13,A14,A15,A1_na,A2_na,A3_na,A4_na,A5_na,A6_na,A7_na,A8_na,A9_na,A10_na,A14_na
477,b,39.17,2.5,y,p,i,h,10.00,f,f,0,t,s,200.0,0,0,0,0,0,0,0,0,0,0,0,0
545,b,44.25,3.0,y,p,d,v,1.29,t,f,0,f,s,0.0,0,0,0,1,0,0,0,0,1,1,1,0


In [18]:
num_vars = X_train.select_dtypes(exclude='O').columns.to_list()
cat_vars = X_train.select_dtypes(include='O').columns.to_list()


pipe = ColumnTransformer(
    [
        ('num_imputer', SimpleImputer(strategy='mean', add_indicator=True), num_vars),
        ('cat_imputer', SimpleImputer(strategy='most_frequent', add_indicator=True), cat_vars)
    ]).set_output(transform='pandas')

In [19]:
X_train_t = pipe.fit_transform(X_train)
X_test_t = pipe.transform(X_test)

In [20]:
X_train_t.sample(2)

,num_imputer__A2,num_imputer__A3,num_imputer__A8,num_imputer__A11,num_imputer__A14,num_imputer__A15,num_imputer__missingindicator_A2,num_imputer__missingindicator_A3,num_imputer__missingindicator_A8,num_imputer__missingindicator_A14,cat_imputer__A1,cat_imputer__A4,cat_imputer__A5,cat_imputer__A6,cat_imputer__A7,cat_imputer__A9,cat_imputer__A10,cat_imputer__A12,cat_imputer__A13,cat_imputer__missingindicator_A1,cat_imputer__missingindicator_A4,cat_imputer__missingindicator_A5,cat_imputer__missingindicator_A6,cat_imputer__missingindicator_A7,cat_imputer__missingindicator_A9,cat_imputer__missingindicator_A10
618,29.58,5.042816,2.651372,1.0,460.0,68.0,0.0,1.0,1.0,0.0,b,u,g,m,v,t,f,t,g,False,False,False,False,False,True,True
443,17.25,3.000000,0.040000,0.0,160.0,40.0,0.0,0.0,0.0,0.0,b,u,g,k,v,f,f,t,g,False,False,False,False,False,False,False


In [28]:
from sklearn.impute import MissingIndicator

indicator = MissingIndicator(features='missing-only').set_output(transform='pandas')

In [29]:
indicator.fit(X_train)

MissingIndicator()

In [30]:
indicator.features_

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 13])

In [31]:
X_train_t = pd.concat(
    [X_train.reset_index(drop=True), indicator.transform(X_train)],
    axis='columns'
)

In [32]:
X_train_t.head()

,A1,A2,A3,A4,A5,A6,A7,A8,A9,A10,A11,A12,A13,A14,A15,missingindicator_A1,missingindicator_A2,missingindicator_A3,missingindicator_A4,missingindicator_A5,missingindicator_A6,missingindicator_A7,missingindicator_A8,missingindicator_A9,missingindicator_A10,missingindicator_A14
0,a,46.08,3.000,u,g,c,v,2.375,t,t,8.0,t,g,396.0,4159.0,False,False,False,False,False,False,False,False,False,False,False
1,a,15.92,NaN,u,g,q,v,NaN,NaN,NaN,0.0,f,g,120.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,b,36.33,2.125,y,p,w,v,0.085,t,t,1.0,f,g,50.0,1187.0,False,False,False,False,False,False,False,False,False,False,False
3,b,22.17,NaN,y,p,ff,ff,NaN,NaN,NaN,0.0,f,g,100.0,0.0,False,False,False,False,False,False,False,False,False,False,False
4,b,57.83,7.040,u,g,m,v,14.000,t,t,6.0,t,g,360.0,1332.0,False,False,True,False,False,False,False,True,True,True,False


In [33]:
X_train

,A1,A2,A3,A4,A5,A6,A7,A8,A9,A10,A11,A12,A13,A14,A15
596,a,46.08,3.000,u,g,c,v,2.375,t,t,8,t,g,396.0,4159
303,a,15.92,NaN,u,g,q,v,NaN,NaN,NaN,0,f,g,120.0,0
204,b,36.33,2.125,y,p,w,v,0.085,t,t,1,f,g,50.0,1187
351,b,22.17,NaN,y,p,ff,ff,NaN,NaN,NaN,0,f,g,100.0,0
118,b,57.83,7.040,u,g,m,v,14.000,t,t,6,t,g,360.0,1332
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
359,a,36.75,NaN,u,g,ff,ff,NaN,NaN,NaN,0,f,g,160.0,0
192,b,41.75,0.960,u,g,x,v,2.500,t,f,0,f,g,510.0,600
629,a,19.58,NaN,u,g,w,v,NaN,NaN,NaN,0,f,g,220.0,5
559,a,22.83,2.290,u,g,q,h,2.290,t,t,7,t,g,140.0,2384
